# ORBIT — Gate 7B: Actor–Critic Research, Algorithm Selection & Architecture Qualification

**Milestone:** Gate 7B (Pre-Gate-8 Experimental Addendum)  
**Controlling Specification:** `ORBIT_IMPLEMENTATION_BRIEF_v1.3.md` (Gate 7B Addendum)  
**Current Production Champion:** Gate 7 AWR Policy (`results/gate7/policy_gate7.pt`)  
**Objective:** Systematically evaluate whether alternative offline RL algorithms (IQL, Discrete CQL, CRR) and alternative critic architectures (Twin Q, Quantile Distributional) or actor designs (Residual MLP) can outperform the Gate 7 production champion.

### Experimental Candidates:
- **Candidate A (Control):** Existing AWR (Gate 7 reproduction).
- **Candidate B (Priority):** Implicit Q-Learning (IQL, Kostrikov et al., 2021) with Expectile Value Critic.
- **Candidate C:** Discrete Conservative Q-Learning (CQL, Kumar et al., 2020) with LogSumExp penalty.
- **Candidate D:** Critic Regularized Regression (CRR, Wang et al., 2020) with binary advantage indicator filter.
- **Critic Qualification:** Scalar Value vs. Twin Q vs. Quantile Distributional Critic.
- **Actor Qualification:** Standard Categorical MLP vs. Parameter-Matched Residual MLP Policy Head.
- **Confirmatory Real Benchmark:** 30 pinned seeds (`5000..5029`) in unmodified Crafter.

### Cell 1: Environment & Hardware Diagnostics

In [ ]:
# Cell 1: Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 75)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 75)
print(f"Python Version:      {platform.python_version()}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:         {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 75)

### Cell 2: Google Drive Mounting & Namespace Setup

In [ ]:
# Cell 2: Google Drive Mounting
import os
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        print("[Drive Mount] Google Drive mounted successfully.")
    else:
        print("[Drive Mount] Google Drive already mounted.")
except ImportError:
    print("[Local Mode] Google Colab drive module not found; running locally.")

drive_orbit_dir = Path('/content/drive/MyDrive/learn-a-world/orbit')
if drive_orbit_dir.parent.exists():
    drive_orbit_dir.mkdir(parents=True, exist_ok=True)
    (drive_orbit_dir / 'results' / 'gate7b').mkdir(parents=True, exist_ok=True)
    print(f"[Persistent Storage] Output directory: {drive_orbit_dir / 'results' / 'gate7b'}")

### Cell 3: Clean Repository Sync & Dependency Setup

In [ ]:
# Cell 3: Repository Sync
%cd /content
import os
from pathlib import Path

repo_path = Path('/content/learn-a-world')
if (repo_path / '.git').exists():
    %cd /content/learn-a-world
    !git fetch origin main
    !git reset --hard origin/main
else:
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
    %cd /content/learn-a-world

!pip install -q crafter matplotlib
print("[Repo Sync] Repository up-to-date on branch main.")

### Cell 4: Checkpoint Verification (Gate 7 Production Champion)

In [ ]:
# Cell 4: Checkpoint Integrity Audit
import hashlib
from pathlib import Path

checkpoints = {
    "Gate 7 Production Champion": Path("results/gate7/policy_gate7.pt"),
    "World Model Champion": Path("results/phase3/world_model_phase3.pt"),
}

for name, p in checkpoints.items():
    if p.exists():
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f"✓ {name:30s}: {p.stat().st_size / (1024*1024):.1f} MB | SHA-256: {sha}")
    else:
        print(f"✗ {name:30s}: NOT FOUND at {p}")

### Cell 5: Execute Gate 7B Research Study (IQL, CQL, CRR & Critic/Actor Evaluation)
Runs all 4 candidates, evaluates critic and actor variations, and runs the locked 30-seed benchmark.

In [ ]:
# Cell 5: Run Gate 7B Research Suite
%cd /content/learn-a-world

!PYTHONPATH=. python3 scripts/run_gate7b_research.py \
    --world-model results/phase3/world_model_phase3.pt \
    --champion-gate7 results/gate7/policy_gate7.pt \
    --replay-dataset data/gate7/augmented_replay_local.npz \
    --heldout-eval data/gate3/eval_heldout_seeds.pt.gz \
    --updates 300 \
    --batch-size 8 \
    --seq-len 8 \
    --benchmark-seeds 30 \
    --max-steps 300 \
    --device auto \
    --output-dir results/gate7b

### Cell 6: Visual Report & Comparative Diagnostic Plots

In [ ]:
# Cell 6: Visual Report & Decision Summary
import json
from IPython.display import Image, display
from pathlib import Path

plot_path = Path("results/gate7b/gate7b_comparison_plots.png")
metrics_path = Path("results/gate7b/gate7b_experiment_metrics.json")

if plot_path.exists():
    display(Image(filename=str(plot_path)))

if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print("=" * 80)
    print(f"GATE 7B FINAL SELECTION: {m.get('selected_architecture')} ({m.get('promotion_status')})")
    print(f"Verdict: {m.get('verdict')}")
    print("=" * 80)
    b = m.get('benchmark_results', {})
    for k, v in b.items():
        print(f"{k:25s} | Return: {v.get('mean_return', 0.0):+.2f} ± {v.get('std_return', 0.0):.2f} | Score: {v.get('crafter_score', 0.0):.2f}% | Steps: {v.get('mean_length', 0.0):.1f}")
    print("=" * 80)